### Building a Chatbot

In this session, we'll go over an example of how to design and implement an LLM powered chatbot. This chatbot will able to have a conversation and remember previous interactions.

Note that this chatbot that we will build will use only language model for conversation. There are several other related concepts that you may be looking for:
- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv() #loading all the .env variables

groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [2]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

/Volumes/Sandisk 1TB/Documents/Generative AI Krish Naik/Building Basic LLM Application Using LCEL(Langchain Expression Language)/My implementation/LCEL/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1345ff550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1345ffbb0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

model.invoke([HumanMessage(content="Hi, My Name is Kaivalya and I am a student!!")])

AIMessage(content='Hello Kaivalya! 👋 Nice to meet you. How can I help you today? Whether it’s a question about your studies, a project you’re working on, or just something you’re curious about, feel free to let me know!', additional_kwargs={'reasoning_content': 'The user greets and introduces themselves. We should respond friendly, perhaps ask how we can help. No disallowed content. Just respond.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 85, 'total_tokens': 173, 'completion_time': 0.189761116, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.004290093, 'prompt_tokens_details': None, 'queue_time': 0.053897552, 'total_time': 0.194051209}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dce5b-8c36-7b10-a058-5cd397a2d24b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_token

In [4]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content='Hi, My Name is Kaivalya and I am a student!!'),
        AIMessage(content='Hello Kaivalya! 👋 Nice to meet you. How can I help you today? Whether it’s a question about your studies, a project you’re working on, or just something you’re curious about, feel free to let me know!'),
        HumanMessage(content='hey what is my name and what do I do?')
    ]
)

AIMessage(content='Your name is **Kaivalya**, and you mentioned that you’re a **student**. Let me know if there’s anything specific you’d like help with—studying, projects, or anything else!', additional_kwargs={'reasoning_content': 'The user asks: "hey what is my name and what do I do?" They previously said: "Hi, My Name is Kaivalya and I am a student!!". So answer: name Kaivalya, student. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 156, 'total_tokens': 258, 'completion_time': 0.223335987, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.00588092, 'prompt_tokens_details': None, 'queue_time': 0.051645275, 'total_time': 0.229216907}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dce5e-7460-7eb2-8c63-9fd76c9765a1-0', tool_calls=[], invalid_tool_calls=[], usage_me

### Message History 
We can use a message history class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future transactions will then load those messages and pass them into the chain as part of the input. 

In [5]:
!pip install langchain_community

In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [20]:
config1={"configurable":{"session_id":"chat1"}}

In [21]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, My Name is Kaivalya and I am a student!!')],
    config=config1
)

In [22]:
with_message_history.invoke(
    [HumanMessage(content='Hi, What is my name??')],
    config=config1
)

AIMessage(content='Your name is **Kaivalya**. 😊 How can I assist you today?', additional_kwargs={'reasoning_content': 'The user asks "Hi, What is my name??". We have context: they told us earlier that their name is Kaivalya. So answer: Kaivalya. Possibly ask if they need anything else.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 392, 'total_tokens': 462, 'completion_time': 0.150802005, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.022274292, 'prompt_tokens_details': None, 'queue_time': 0.050514771, 'total_time': 0.173076297}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dce71-426c-7ad3-adfc-3e73f99cee68-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 392, 'output_tokens': 70, 'total_tokens': 462, 'output_token_details': {'reasoning': 44}})

In [23]:
# Change the config
config2={"configurable":{"session_id":"chat2"}}

In [26]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, What is my name??')],
    config=config2
)
response.content

'I don’t have your name from our previous messages. If you’d like me to address you by name, just tell me what you’d like to be called!'

In [28]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, My Name is Kaivalya')],
    config=config2
)
response.content

'Got it—nice to meet you, Kaivalya! What can I help you with today?'

In [29]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, My Name is ?')],
    config=config2
)
response.content

'You introduced yourself earlier as **Kaivalya**. If that’s still the name you’d like me to use, just let me know—otherwise feel free to tell me any other name you prefer!'

## Prompt Templates
- Prompt templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a messaage, which we are passing to the LLM. Let's now make it bit complicated. First, let's add in a system message with some custom instructions (but still taking messages as input).Next, we'll add in more input besides just the messages. 

In [31]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [33]:
chain.invoke({"messages":[HumanMessage(content='My name is Kaivalya!!')]}).content

"Nice to meet you, Kaivalya! If there's anything I can help you with, just let me know."

In [34]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [36]:
config3 = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content='My name is Kaivalya!!')],
    config=config3
)

response

AIMessage(content='Got it—nice to meet you, Kaivalya! How can I help you today?', additional_kwargs={'reasoning_content': 'The user repeats "My name is Kaivalya!!". The assistant should respond acknowledging. Possibly ask if they want anything. Keep friendly.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 130, 'total_tokens': 187, 'completion_time': 0.122668203, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.006560689, 'prompt_tokens_details': None, 'queue_time': 0.050435545, 'total_time': 0.129228892}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a19b1544c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dce80-6e84-73d3-93cd-ec8cd31f4ee1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 57, 'total_tokens': 187, 'output_token_details': {'reasoning': 29}})

In [38]:
response = with_message_history.invoke(
    [HumanMessage(content='What is my name?')],
    config=config3
)

response

AIMessage(content='You told me earlier that your name is **Kaivalya**. If that’s correct, great—nice to meet you, Kaivalya! If you’d like to use a different name, just let me know.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" Previously they said "My name is Kaivalya!!". Then later they said "My name is ??". Now ask again. We should answer based on conversation: The name is Kaivalya. But they might be confused. We can politely ask for clarification, but we have prior info. We can answer: you said your name is Kaivalya. If that\'s not correct, let me know.'}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 224, 'total_tokens': 369, 'completion_time': 0.304792202, 'completion_tokens_details': {'reasoning_tokens': 91}, 'prompt_time': 0.010328947, 'prompt_tokens_details': None, 'queue_time': 0.051837487, 'total_time': 0.315121149}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ce855234b5', 'service_tier': 'on_dem

In [39]:
# Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.Answer all questions to the best of your ability in {language}."
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [40]:
response = chain.invoke({"messages":[HumanMessage(content="Hi my name is Kaivalya")],"language":"Hindi"})
response.content

'नमस्ते कैलव्या! आपसे मिलकर ख़ुशी हुई। आज मैं आपकी कैसे मदद कर सकता हूँ?'

#### Let's now wrap this more complicated chain in message history class. This time, because there are multiple keys in the input, we need to specify the correct key to use save the chat history.

In [41]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [42]:
config4 = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hi my name is Kaivalya")],"language":"hindi"},
    config=config4
)
response.content

'नमस्ते! आपका स्वागत है, काइवेल्य। आपसे मिलकर खुशी हुई। आप कैसे हैं?'

In [43]:
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="What's my name")],"language":"Hindi"},
    config=config4
)
response.content

'आपका नाम काइवेल्य है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the lists of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages': helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters if we want to keep the system message and whether to allow partial message.

In [45]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on='human'
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [46]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)
response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do I like?")],
    "language":"English"
    }
)
response.content

'I’m not sure which ice‑cream flavor you prefer—could you tell me a bit about the kinds you usually enjoy?'

In [47]:
response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What math problem did i asked?")],
    "language":"English"
    }
)
response.content

'You asked about the addition problem **2\u202f+\u202f2**.'

In [52]:
# Let's wrap this thing in message history 
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config5 = {'configurable':{'session_id':'chat5'}}

In [55]:
response = with_message_history.invoke(
    {
    "messages":messages + [HumanMessage(content="What's my name?")],
    "language":"English"
    },
    config=config5
)
response.content

'I don’t actually know your name—unless you’ve told me before in this conversation, I don’t have that information. If you’d like, feel free to share it, and I can address you by name from now on!'

In [57]:
response = with_message_history.invoke(
    {
    "messages":messages + [HumanMessage(content="What math problem did i asked?")],
    "language":"English"
    },
    config=config5
)
response.content

'You asked for the result of the addition problem:\n\n**“What is\u202f2\u202f+\u202f2?”**'

In [ ]:
aaa